# 🍁 Charity Intelligence Canada — Web Scraper
### World Vision Competitive Set Refresh

**What this notebook does, end to end:**

| Step | Cell | What happens |
|------|------|--------------|
| 0 | Setup | Install libraries, set constants & competitive set |
| 1 | robots.txt | Confirm we are allowed to scrape |
| 2 | HTTP Client | Polite, cached, retry-safe client |
| 3 | Inspect | Look at one live profile before scraping everything |
| 4 | Parser | Full parser — every workbook column |
| 5 | Verify | Parse World Vision, check every field |
| 6 | Discover | A-Z listing → full charity index (optional) |
| 7 | Batch | Scrape all charities in competitive set |
| 8 | Build | Assemble master DataFrames |
| 9 | Validate | Data-quality checks against contract thresholds |
| 10 | Save | CSV + Excel workbook + Sofia upload tab |
| 11 | Preview | Review what was scraped |
| 12 | Next steps | Maintenance guide |

> **Attribution:** Source: Charity Intelligence (charityintelligence.ca)  
> **Politeness:** 2-second delay · robots.txt honoured · identifying User-Agent  
> Run everything with **Kernel → Restart & Run All**


## Cell 1 — Setup & Imports

In [ ]:
# ── Install any missing libraries ──────────────────────────────────
import subprocess, sys

REQUIRED = ["requests", "beautifulsoup4", "lxml", "pandas", "openpyxl", "tqdm"]
for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-","_").split("[")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"  installed {pkg}")

# ── Standard imports ────────────────────────────────────────────────
import re, time, json, hashlib, logging, os
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.robotparser import RobotFileParser
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s"
)
log = logging.getLogger("ci_scraper")

# ── Constants ───────────────────────────────────────────────────────
# ⚠️  PUT YOUR EMAIL BELOW — it goes in the User-Agent (polite scraping)
CONTACT_EMAIL  = os.getenv("CI_CONTACT_EMAIL", "your.email@worldvision.ca")

BASE_URL       = "https://www.charityintelligence.ca"
LISTING_URL    = BASE_URL + "/charity-profiles/a-z-charity-listing"
REQUEST_DELAY  = 2.0            # seconds between live requests
MAX_AGE_DAYS   = 7.0            # re-use cached HTML if younger than this
CACHE_DIR      = Path("cache/html")
OUTPUT_DIR     = Path("outputs")
PARSER_VERSION = "1.2.0"

# ── Competitive set  ────────────────────────────────────────────────
# Add / remove charities here. profile_url comes from the CI website.
COMPETITIVE_SET = [
    {"charity_name": "World Vision Canada",
     "profile_url": BASE_URL + "/charity-details/33-world-vision-canada"},
    {"charity_name": "Canadian Red Cross",
     "profile_url": BASE_URL + "/charity-details/6-canadian-red-cross"},
    {"charity_name": "Doctors Without Borders",
     "profile_url": BASE_URL + "/charity-details/14-medecins-sans-frontieres"},
    {"charity_name": "Samaritan's Purse",
     "profile_url": BASE_URL + "/charity-details/7-samaritans-purse-canada"},
    {"charity_name": "UNICEF Canada",
     "profile_url": BASE_URL + "/charity-details/38-unicef-canada"},
    {"charity_name": "Plan International Canada",
     "profile_url": BASE_URL + "/charity-details/21-plan-canada"},
    {"charity_name": "Save the Children Canada",
     "profile_url": BASE_URL + "/charity-details/22-save-the-children-canada"},
    {"charity_name": "CARE Canada",
     "profile_url": BASE_URL + "/charity-details/5-care-canada"},
    {"charity_name": "Oxfam Canada",
     "profile_url": BASE_URL + "/charity-details/31-oxfam-canada"},
    {"charity_name": "Canadian Cancer Society",
     "profile_url": BASE_URL + "/charity-details/10-canadian-cancer-society"},
]

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅  Setup complete.")
print(f"    Competitive set : {len(COMPETITIVE_SET)} charities")
print(f"    Contact email   : {CONTACT_EMAIL}")
print(f"    Cache dir       : {CACHE_DIR.resolve()}")
print(f"    Output dir      : {OUTPUT_DIR.resolve()}")


## Cell 2 — Confirm robots.txt Allows Scraping

In [ ]:
USER_AGENT = f"wv-competitive-refresh/{PARSER_VERSION} (+{CONTACT_EMAIL})"

robots = RobotFileParser()
robots.set_url(BASE_URL + "/robots.txt")
robots.read()

test_urls = [
    LISTING_URL,
    BASE_URL + "/charity-details/33-world-vision-canada",
]

print(f"User-Agent: {USER_AGENT}\n")
all_allowed = True
for url in test_urls:
    allowed = robots.can_fetch(USER_AGENT, url)
    icon    = "✅ ALLOWED" if allowed else "🚫 BLOCKED"
    print(f"  {icon}  →  {url}")
    if not allowed:
        all_allowed = False

cd = robots.crawl_delay(USER_AGENT) or robots.crawl_delay("*")
if cd:
    REQUEST_DELAY = max(REQUEST_DELAY, float(cd))
    print(f"\n  robots.txt crawl-delay={cd}s — using {REQUEST_DELAY}s")
else:
    print(f"\n  No crawl-delay specified — using default {REQUEST_DELAY}s")

if not all_allowed:
    raise RuntimeError("robots.txt blocks our User-Agent. Do NOT proceed.")

print("\n✅  Scraping is permitted.")


## Cell 3 — HTTP Client (Polite · Cached · Retry-safe)

In [ ]:
class CIClient:
    """
    Polite HTTP client for the Charity Intelligence website.

    Features
    --------
    - Identifying User-Agent (name + contact email)
    - Exponential-backoff retries on 429 / 5xx
    - On-disk HTML cache:  re-runs use cache, no extra requests
    - Rate limiting:  REQUEST_DELAY seconds between live requests
    """

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": USER_AGENT,
            "Accept":     "text/html,application/xhtml+xml",
        })
        retry = Retry(
            total=4,
            backoff_factor=2,
            status_forcelist=[429, 500, 502, 503, 504],
            respect_retry_after_header=True,
        )
        self.session.mount("https://", HTTPAdapter(max_retries=retry))
        self._last_request = 0.0

    def _throttle(self):
        wait = REQUEST_DELAY - (time.monotonic() - self._last_request)
        if wait > 0:
            time.sleep(wait)
        self._last_request = time.monotonic()

    def get(self, url: str, cache_name: str,
            max_age_days: float = MAX_AGE_DAYS) -> Optional[str]:
        """Return HTML string.  Serve from cache if file is fresh enough."""
        path = CACHE_DIR / cache_name
        if path.exists():
            age = (time.time() - path.stat().st_mtime) / 86400
            if age < max_age_days:
                return path.read_text(encoding="utf-8")   # cache hit — no request
        self._throttle()
        try:
            r = self.session.get(url, timeout=30)
            r.raise_for_status()
            path.write_text(r.text, encoding="utf-8")
            return r.text
        except requests.RequestException as e:
            log.error("GET failed  %s  —  %s", url, e)
            return None


client = CIClient()
print("✅  HTTP client ready.")


## Cell 4 — Inspect One Live Profile (World Vision)

Fetch World Vision once and confirm the page structure before we build the full parser.

In [ ]:
WV_URL   = BASE_URL + "/charity-details/33-world-vision-canada"
wv_html  = client.get(WV_URL, "33-world-vision-canada.html")

if not wv_html:
    raise RuntimeError("Could not fetch World Vision profile. Check internet connection.")

soup      = BeautifulSoup(wv_html, "lxml")
full_text = soup.get_text("\n", strip=True)

og_title  = soup.find("meta", attrs={"property": "og:title"})
star_m    = re.search(r"Charity Rating:\s*(\d)/5", wv_html)
tables    = soup.find_all("table")
updated_m = re.search(r"Updated on\s+([A-Za-z]+ \d{1,2}, \d{4})", full_text)

print("── Page diagnostics ───────────────────────────────────────────────────")
print(f"  og:title          : {og_title['content'] if og_title else 'NOT FOUND'}")
print(f"  Star rating       : {star_m.group() if star_m else 'NOT FOUND (login wall?)'}")
print(f"  Tables on page    : {len(tables)}")
print(f"  Last updated      : {updated_m.group(1) if updated_m else 'NOT FOUND'}")
print(f"  HTML size         : {len(wv_html):,} chars")

print("\n── Rating tile values ─────────────────────────────────────────────────")
TILE_HEADINGS = [
    "FINANCIAL TRANSPARENCY", "RESULTS REPORTING",
    "DEMONSTRATED IMPACT", "NEED FOR FUNDING", "CENTS TO THE CAUSE",
]
for heading in TILE_HEADINGS:
    h_node = soup.find(string=re.compile(heading, re.I))
    if h_node:
        prev = h_node.parent.find_previous(
            string=lambda s: (s and s.strip() and len(s.strip()) < 15
                              and s.parent.name not in ("script","style","a"))
        )
        print(f"  {heading:<30}: {prev.strip() if prev else '(icon-only — no text value)'}")

print("\n── Table headers ──────────────────────────────────────────────────────")
for i, t in enumerate(tables):
    first_row = t.find("tr")
    if first_row:
        headers = [c.get_text(" ", strip=True)[:55]
                   for c in first_row.find_all(["th","td"])]
        print(f"  Table {i}: {headers}")

print("\n── Address block (first 6 lines) ──────────────────────────────────────")
h1s = [h.get_text(" ", strip=True) for h in soup.find_all("h1")
       if h.get_text(strip=True).lower() not in ("my anchor","")]
print(f"  Charity name (h1) : {h1s[0] if h1s else 'NOT FOUND'}")
reg_m = re.search(r"Charitable Reg\.?\s*#:?\s*([\d\s]+RR\s?\d{4})", full_text)
print(f"  Registration #    : {reg_m.group(1).strip() if reg_m else 'NOT FOUND'}")


## Cell 5 — Full Profile Parser

All mapping tables + helper functions + `parse_profile()`.

In [ ]:
# ────────────────────────────────────────────────────────────────────
# Label → contract column name  (normalised lowercase label as key)
# ────────────────────────────────────────────────────────────────────
RATIO_MAP = {
    "fundraising & admin cost % revenue":       "fundraising_admin_cost_pct_revenue",
    "fundraising and admin cost % revenue":     "fundraising_admin_cost_pct_revenue",
    "administrative costs as % of revenues":    "admin_cost_pct_revenue",
    "fundraising costs as % of donations":      "fundraising_cost_pct_donations",
    "total overhead spending":                  "total_overhead_pct",
    "program cost coverage":                    "program_cost_coverage_pct",
    "program cost coverage (%)":                "program_cost_coverage_pct",
}

FIN_MAP = {
    "canadian donations":                       "canadian_donations",
    "donations":                                "canadian_donations",
    "international donations":                  "international_donations",
    "goods in kind":                            "gifts_in_kind",
    "government funding":                       "government_funding",
    "special events":                           "special_events",
    "fees for service":                         "fees_for_service",
    "business activities":                      "business_activities",
    "lotteries":                                "lotteries",
    "investment income":                        "investment_income",
    "other income":                             "other_income",
    "total revenues":                           "total_revenues",
    "program spending":                         "program_spending",
    "program costs":                            "program_spending",
    "fundraising costs":                        "fundraising_costs",
    "administrative costs":                     "administrative_costs",
    "total spending":                           "total_spending",
    "grant spending":                           "grant_spending",
    "grants":                                   "grant_spending",
    "donated goods exp":                        "gifts_in_kind_spending",
    "donated goods expense":                    "gifts_in_kind_spending",
    "other costs":                              "other_costs",
    "program spending - canada":                "program_spending_canada",
    "program costs - canada":                   "program_spending_canada",
    "program spending - international":         "program_spending_international",
    "program costs - international":            "program_spending_international",
    "funding reserves":                         "funding_reserves",
    "capital spending":                         "capital_spending",
    "cash flow from operations":                "cash_flow_from_ops",
}

# Compensation band label patterns → column name
BAND_PATTERNS = [
    (r"\$350k\+|350,000.*over|\$350k \+",        "comp_band_350k_plus"),
    (r"\$300k.*\$?350k|300,000.*350",               "comp_band_300k_350k"),
    (r"\$250k.*\$?300k|250,000.*300",               "comp_band_250k_300k"),
    (r"\$200k.*\$?250k|200,000.*250",               "comp_band_200k_250k"),
    (r"\$160k.*\$?200k|160,000.*200",               "comp_band_160k_200k"),
    (r"\$120k.*\$?160k|120,000.*160",               "comp_band_120k_160k"),
    (r"\$80k.*\$?120k|80,000.*120",                 "comp_band_80k_120k"),
    (r"\$40k.*\$?80k|40,000.*80",                   "comp_band_40k_80k"),
    (r"<.*\$?40k|under.*40|<.*40,000",               "comp_band_lt_40k"),
]

# ────────────────────────────────────────────────────────────────────
# Helper functions
# ────────────────────────────────────────────────────────────────────
def norm(s: str) -> str:
    """Normalise a label string for dictionary lookup."""
    s = s.lower().replace("\xa0"," ").replace("\u2013","-").replace("\u2014","-")
    s = re.sub(r"\(\s*%\s*\)", "", s)
    return re.sub(r"\s+", " ", s).strip(" :")

def parse_number(s: Optional[str]) -> Optional[float]:
    """
    Convert a CI money/ratio string to a Python float.

    Examples
    --------
    '185,269'    ->  185_269_000   (CI financial figures are in $000s)
    '(6,493)'    ->  -6_493_000    (parentheses = negative)
    '5.3%'       ->  5.3           (percentage: no multiplier)
    '—' / 'n/a'  ->  None
    """
    if not s:
        return None
    t = s.strip().replace("\u2212", "-").replace("\xa0", " ")
    if re.fullmatch(r"[\-\u2014\u2013]|n/?a|nm", t, re.I):
        return None
    neg    = t.startswith("(") and t.endswith(")")
    is_pct = "%" in t
    t = re.sub(r"[()$,%\s]", "", t)
    try:
        v = float(t)
    except ValueError:
        return None
    if neg:
        v = -v
    # CI financial statements are published in $000s — multiply to whole CAD
    return v if is_pct else v * 1_000

def match_band(label: str) -> Optional[str]:
    """Return the comp_band_* column name for a salary-band label, or None."""
    label = label.lower().replace("\u2013","-").replace("\u2014","-")
    for pat, col in BAND_PATTERNS:
        if re.search(pat, label, re.I):
            return col
    return None

def table_to_rows(table) -> list:
    """Extract a list-of-lists from a <table> element."""
    return [
        [c.get_text(" ", strip=True) for c in tr.find_all(["th","td"])]
        for tr in table.find_all("tr")
        if tr.find_all(["th","td"])
    ]

def extract_years(header_row: list) -> list:
    """Pull fiscal years (integers) from the header row of a financial table."""
    years = []
    for cell in header_row[1:]:
        m = re.search(r"(?:19|20)\d{2}", cell)
        years.append(int(m.group()) if m else None)
    return years

# ────────────────────────────────────────────────────────────────────
# Main parser
# ────────────────────────────────────────────────────────────────────
def parse_profile(html: str, url: str, scraped_at: str) -> dict:
    """
    Parse one CI charity profile page into structured data.

    Parameters
    ----------
    html       : raw page HTML
    url        : canonical profile URL (used to extract charity_id)
    scraped_at : ISO-8601 UTC timestamp

    Returns
    -------
    dict with keys:
      'profile'          flat dict  – one row in ci_metrics.csv
      'financials_long'  list[dict] – 3 years × all metrics (long format)
      'warnings'         list[str]  – non-fatal issues for review
    """
    soup     = BeautifulSoup(html, "lxml")
    full     = soup.get_text("\n", strip=True)
    warnings = []

    # ── Identity ────────────────────────────────────────────────────
    m = re.search(r"/charity-details/(\d+)-([^/?#\s]+)", url)
    if not m:
        raise ValueError(f"Not a charity-details URL: {url}")

    p = {
        "charity_id"     : int(m.group(1)),
        "slug"           : m.group(2),
        "profile_url"    : url.split("?")[0],
        "scraped_at"     : scraped_at,
        "parser_version" : PARSER_VERSION,
    }

    # ── Charity name ────────────────────────────────────────────────
    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        p["charity_name"] = og["content"].strip()
    else:
        h1s = [h.get_text(" ", strip=True) for h in soup.find_all("h1")
               if h.get_text(strip=True).lower() not in ("my anchor","")]
        p["charity_name"] = h1s[0] if h1s else None

    # ── Sector ──────────────────────────────────────────────────────
    sa = soup.select_one('a[href*="sectorsearch"]')
    p["sector"] = sa.get_text(" ", strip=True) if sa else None

    # ── Star rating ──────────────────────────────────────────────────
    rm = re.search(r"Charity Rating:\s*(\d)/5", html)
    p["star_rating"] = int(rm.group(1)) if rm else None
    if p["star_rating"] is None:
        warnings.append("star_rating not found — may be behind login wall")

    # ── Rating tiles ─────────────────────────────────────────────────
    # The short text value sits immediately BEFORE each tile heading.
    TILES = {
        "financial transparency" : "financial_transparency",
        "results reporting"      : "results_reporting_grade",
        "demonstrated impact"    : "demonstrated_impact_rating",
        "cents to the cause"     : "_cents_raw",
    }
    for tag in soup.find_all(["h2","h3","h4","h5"]):
        label = norm(tag.get_text(" ", strip=True))
        field = TILES.get(label)
        if not field:
            continue
        prev = tag.find_previous(
            string=lambda s: (
                s and isinstance(s, str) and s.strip()
                and 0 < len(s.strip()) <= 15
                and s.parent.name not in ("script","style","a","nav","li")
            )
        )
        p[field] = prev.strip() if prev else None

    p["cents_to_the_cause_pct"] = parse_number(p.pop("_cents_raw", None))
    for f in TILES.values():
        p.setdefault(f.lstrip("_"), None)

    # ── Need for Funding (numeric) ────────────────────────────────────
    for key in ("need_for_funding_program_costs",
                "need_for_funding_reserve_funds",
                "need_for_funding_endowments"):
        p[key] = None

    nff_node = soup.find(string=re.compile(r"NEED FOR FUNDING", re.I))
    if nff_node:
        container = nff_node.parent
        for _ in range(6):
            container = container.parent
            nums = [float(x) for x in re.findall(r"\b\d+\.?\d*\b", container.get_text())
                    if 0 <= float(x) <= 10_000]
            if nums:
                p["need_for_funding_program_costs"] = nums[0] if len(nums) > 0 else None
                p["need_for_funding_reserve_funds"]  = nums[1] if len(nums) > 1 else None
                p["need_for_funding_endowments"]     = nums[2] if len(nums) > 2 else None
                break
    if not p["need_for_funding_program_costs"]:
        warnings.append("need_for_funding: no numeric value — likely icon-only on this profile")

    # ── Badges (only inside <main>, not global nav) ──────────────────
    main_el = soup.find("main") or soup
    links   = [a.get_text(" ", strip=True) for a in main_el.find_all("a")]
    p["is_top_100_rated"]      = any(re.search(r"Top 100 Rated",      t, re.I) for t in links)
    p["is_top_impact_charity"] = any(re.search(r"Top.*Impact Charit", t, re.I) for t in links)

    # ── Last updated & registration number ──────────────────────────
    upd = re.search(r"Updated on\s+([A-Za-z]+\s+\d{1,2},\s*\d{4})", full)
    p["profile_last_updated"] = None
    if upd:
        try:
            p["profile_last_updated"] = datetime.strptime(
                re.sub(r"\s+"," ", upd.group(1)), "%B %d, %Y"
            ).date().isoformat()
        except ValueError:
            warnings.append(f"Could not parse update date: {upd.group(1)!r}")

    reg = re.search(r"Charitable\s+Reg\.?\s*#:?\s*([\d\s]+RR\s?\d{4})", full, re.I)
    p["charitable_reg_number"] = (
        re.sub(r"\s+","", reg.group(1)).upper() if reg else None
    )

    # ── Staff & compensation ──────────────────────────────────────────
    sm = re.search(r"Full-time staff[:\s]+(\d[\d,]*)", full)
    p["full_time_staff"] = int(sm.group(1).replace(",","")) if sm else None

    cm = re.search(r"Avg\.?\s*compensation[:\s]+\$?([\d,]+)", full)
    p["avg_compensation_cad"] = float(cm.group(1).replace(",","")) if cm else None

    # ── Financial tables ──────────────────────────────────────────────
    financials_long: list = []
    p["fiscal_year_latest"] = p["year_end_month"] = None

    for table in soup.find_all("table"):
        rows = table_to_rows(table)
        if not rows or len(rows[0]) < 2:
            continue
        head0 = norm(rows[0][0])

        # ── Compensation bands ──────────────────────────────────────
        if any(match_band(r[0]) for r in rows if r):
            for r in rows:
                col = match_band(r[0])
                if col and len(r) > 1:
                    v = re.sub(r"[^\d]","", r[1])
                    p[col] = int(v) if v else 0
            continue

        # ── Financial ratios table ──────────────────────────────────
        if "financial ratios" in head0:
            yem = re.search(r"ending\s+([A-Za-z]+)", rows[0][0])
            p["year_end_month"] = yem.group(1).capitalize() if yem else None
            years = extract_years(rows[0])
            p["fiscal_year_latest"] = max((y for y in years if y), default=None)

            for r in rows[1:]:
                lbl = norm(r[0])
                col = RATIO_MAP.get(lbl)
                for yr, txt in zip(years, r[1:]):
                    if yr is None:
                        continue
                    val = parse_number(txt)
                    # Ratios are percentages — undo the $000s multiplier
                    if val is not None and abs(val) > 500:
                        val = val / 1_000
                    financials_long.append({
                        "fiscal_year" : yr,
                        "group"       : "ratio",
                        "metric"      : col or f"unmapped::{lbl[:40]}",
                        "value"       : val,
                        "is_mapped"   : col is not None,
                    })
                # Latest FY → flat profile
                if col and years and years[0] and len(r) > 1:
                    v = parse_number(r[1])
                    p[col] = (v / 1_000) if (v and abs(v) > 500) else v

        # ── Summary financial statements table ──────────────────────
        elif "summary financial" in head0:
            years = extract_years(rows[0])
            if p["fiscal_year_latest"] is None:
                p["fiscal_year_latest"] = max((y for y in years if y), default=None)

            for r in rows[1:]:
                lbl = norm(r[0])
                col = FIN_MAP.get(lbl)
                if not col and lbl:
                    warnings.append(f"Unmapped financial label: '{r[0]}'")
                for yr, txt in zip(years, r[1:]):
                    if yr is None:
                        continue
                    financials_long.append({
                        "fiscal_year" : yr,
                        "group"       : "financial_statement",
                        "metric"      : col or f"unmapped::{lbl[:40]}",
                        "value"       : parse_number(txt),
                        "is_mapped"   : col is not None,
                    })
                # Latest FY → flat profile
                if col and years and years[0] and len(r) > 1:
                    p[col] = parse_number(r[1])

    if not financials_long:
        warnings.append("No financial tables found — check the cached HTML")

    # ── Semantic hash ─────────────────────────────────────────────────
    # Excludes scrape timestamp so the hash only changes when data changes.
    core = {k: v for k, v in p.items()
            if k not in ("scraped_at","parser_version")}
    p["semantic_hash"] = hashlib.sha256(
        json.dumps(core, sort_keys=True, default=str).encode()
    ).hexdigest()

    return {
        "profile"         : p,
        "financials_long" : financials_long,
        "warnings"        : warnings,
    }


print("✅  Parser ready.")
print(f"    RATIO_MAP  : {len(RATIO_MAP)} labels")
print(f"    FIN_MAP    : {len(FIN_MAP)} labels")
print(f"    BAND_PATTERNS : {len(BAND_PATTERNS)} salary bands")


## Cell 6 — Parse & Verify World Vision (Single Profile Test)

Re-uses the HTML already cached in Cell 4 — no second request.

In [ ]:
scraped_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
result_wv  = parse_profile(wv_html, WV_URL, scraped_at)

p  = result_wv["profile"]
fl = result_wv["financials_long"]
w  = result_wv["warnings"]

div = "═" * 65
print(div)
print(f"  Charity  : {p.get('charity_name')}")
print(f"  Sector   : {p.get('sector')}")
print(f"  FY       : {p.get('fiscal_year_latest')}  "
      f"(year-end: {p.get('year_end_month')})")
print(f"  Updated  : {p.get('profile_last_updated')}")
print(f"  Reg #    : {p.get('charitable_reg_number')}")
print(div)

print("\n── CI Ratings ──────────────────────────────────────────────────")
fields = {
    "star_rating"              : "Star rating",
    "financial_transparency"   : "Financial transparency",
    "results_reporting_grade"  : "Results reporting",
    "demonstrated_impact_rating": "Demonstrated impact",
    "cents_to_the_cause_pct"   : "Cents to the cause (%)",
    "is_top_100_rated"         : "Top 100 rated",
    "is_top_impact_charity"    : "Top impact charity",
}
for key, label in fields.items():
    val = p.get(key)
    suffix = "%" if "pct" in key and val is not None else ""
    print(f"  {label:<30}: {val}{suffix}")

print("\n── Financial Ratios (Latest FY) ────────────────────────────────")
for col, label in [
    ("admin_cost_pct_revenue",        "Admin cost % revenues"),
    ("fundraising_cost_pct_donations","Fundraising cost % donations"),
    ("total_overhead_pct",            "Total overhead"),
    ("program_cost_coverage_pct",     "Program cost coverage"),
]:
    val = p.get(col)
    print(f"  {label:<35}: {f'{val:.1f}%' if val is not None else 'N/A'}")

print("\n── Summary Financials ($CAD, Latest FY) ────────────────────────")
fin_cols = [
    ("canadian_donations",    "Canadian donations"),
    ("international_donations","International donations"),
    ("gifts_in_kind",         "Gifts in kind"),
    ("government_funding",    "Government funding"),
    ("total_revenues",        "Total revenues"),
    ("program_spending",      "Program spending"),
    ("fundraising_costs",     "Fundraising costs"),
    ("administrative_costs",  "Administrative costs"),
    ("total_spending",        "Total spending"),
    ("funding_reserves",      "Funding reserves"),
    ("cash_flow_from_ops",    "Cash flow from ops"),
]
for col, label in fin_cols:
    val = p.get(col)
    if val is not None:
        print(f"  {label:<30}: ${val:>18,.0f}")
    else:
        print(f"  {label:<30}: {'N/A':>19}")

print("\n── Staff ────────────────────────────────────────────────────────")
ft = p.get("full_time_staff")
ac = p.get("avg_compensation_cad")
print(f"  Full-time staff     : {f'{ft:,}' if ft else 'N/A'}")
print(f"  Avg. compensation   : {f'${ac:,.0f}' if ac else 'N/A'}")

band_cols = [c for c in p if c.startswith("comp_band_") and p[c]]
if band_cols:
    print("  Top-10 salary bands :")
    for col in band_cols:
        label = col.replace("comp_band_","").replace("_"," ").replace("lt","<")
        print(f"    {label:<20}: {p[col]}")

print("\n── Long financials ──────────────────────────────────────────────")
print(f"  Total rows          : {len(fl)}")
print(f"  Unmapped labels     : {sum(1 for r in fl if not r['is_mapped'])}")
years_found = sorted({r['fiscal_year'] for r in fl if r['fiscal_year']})
print(f"  Fiscal years        : {years_found}")

print("\n── Warnings ─────────────────────────────────────────────────────")
for warning in (w or ["(none)"]):
    print(f"  ⚠️  {warning}")

print("\n── Semantic hash ────────────────────────────────────────────────")
print(f"  {p.get('semantic_hash')}")
print("\n✅  Verification complete — check values match the CI website.")


## Cell 7 — Discover Full Charity Index (Optional)

Walk the A-Z listing to enumerate all ~800 charities.
Set `DISCOVER_ALL = True` to run this. Otherwise the notebook uses `COMPETITIVE_SET` from Cell 1.

In [ ]:
DISCOVER_ALL = False   # ← set to True to enumerate all ~800 charities

LINK_RE   = re.compile(r"/charity-details/(\d+)-([^/?#\s]+)")
RATING_RE = re.compile(r"Charity Rating:\s*(\d)/5")

def parse_listing_page(html: str) -> list:
    """Extract charity rows from one A-Z listing page."""
    soup = BeautifulSoup(html, "lxml")
    seen, rows = set(), []
    for a in soup.select('a[href*="/charity-details/"]'):
        m = LINK_RE.search(a.get("href",""))
        if not m or int(m.group(1)) in seen:
            continue
        cid = int(m.group(1)); seen.add(cid)
        li  = a.find_parent("li") or a
        blob = str(li)
        rating = RATING_RE.search(blob)
        strings = [s.strip() for s in li.stripped_strings
                   if s.strip() and s.strip() != a.get_text(strip=True)]
        rows.append({
            "charity_id"            : cid,
            "slug"                  : m.group(2),
            "charity_name"          : a.get_text(" ", strip=True),
            "profile_url"           : urljoin(BASE_URL, a["href"].split("?")[0]),
            "star_rating_listing"   : int(rating.group(1)) if rating else None,
            "tagline"               : strings[0] if strings else None,
            "sector_listing"        : strings[1] if len(strings) > 1 else None,
            "location"              : strings[2] if len(strings) > 2 else None,
            "discovered_at"         : datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        })
    return rows

if DISCOVER_ALL:
    index, page = {}, 0
    print("Discovering charities from A-Z listing (~3 min uncached)...")
    with tqdm(desc="Listing pages") as pbar:
        while page < 200:   # safety limit
            url_page     = f"{LISTING_URL}?start={page * 20}"
            html_listing = client.get(url_page, f"listing_p{page:03d}.html", max_age_days=1)
            if not html_listing:
                print(f"\n⚠️  Failed on page {page}. Stopping."); break
            new = [r for r in parse_listing_page(html_listing)
                   if r["charity_id"] not in index]
            if not new:
                print(f"\n✅  No new charities on page {page}. Done."); break
            for r in new:
                index[r["charity_id"]] = r
            pbar.update(1)
            pbar.set_postfix(total=len(index), new_this_page=len(new))
            page += 1

    df_index = pd.DataFrame(list(index.values()))
    df_index.to_csv(OUTPUT_DIR / "charity_index.csv", index=False)
    print(f"\n✅  {len(df_index):,} charities discovered.")
    print(f"    Saved → {OUTPUT_DIR}/charity_index.csv")
    display(df_index[["charity_id","charity_name","star_rating_listing",
                       "sector_listing","location"]].head(15))
else:
    print("ℹ️  DISCOVER_ALL = False — using COMPETITIVE_SET from Cell 1.")
    print("    Set DISCOVER_ALL = True to enumerate all ~800 charities.")
    df_index = pd.DataFrame(COMPETITIVE_SET)


## Cell 8 — Batch Scrape All Charities

Polite, cached, resumable. Re-running reuses `cache/html/` automatically.

In [ ]:
work_list              = df_index.to_dict("records")
all_results: list      = []
errors:      list      = []
run_at                 = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

est_min = len(work_list) * REQUEST_DELAY / 60
print(f"Scraping {len(work_list)} charities")
print(f"  Delay       : {REQUEST_DELAY}s between live requests")
print(f"  Est. time   : ~{est_min:.0f} min uncached  (much faster if cache exists)")
print(f"  Cache dir   : {CACHE_DIR.resolve()}")
print("-" * 60)

for row in tqdm(work_list, desc="Profiles", unit="charity"):
    url = row["profile_url"]
    m   = re.search(r"/charity-details/(\d+)-", url)
    cid = int(m.group(1)) if m else None
    cache_name = f"{cid}.html" if cid else re.sub(r"[^a-z0-9]","_", url[-30:]) + ".html"

    page_html = client.get(url, cache_name)
    if not page_html:
        errors.append({"url": url, "error": "fetch_failed"})
        continue

    cache_path = CACHE_DIR / cache_name
    scraped_at = datetime.fromtimestamp(
        cache_path.stat().st_mtime, tz=timezone.utc
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    try:
        result = parse_profile(page_html, url, scraped_at)
        all_results.append(result)
    except Exception as exc:
        errors.append({"url": url, "error": repr(exc)})

# ── Summary ────────────────────────────────────────────────────────
n_ok  = len(all_results)
n_err = len(errors)
print(f"\n✅  Parsed  : {n_ok}/{len(work_list)}")
print(f"   Errors  : {n_err}")
if errors:
    print("\n⚠️  Error details:")
    for e in errors:
        print(f"    {e['url']}")
        print(f"      → {e['error']}")

# Warn if too many warnings across profiles
all_warnings = [(r["profile"].get("charity_name"), w)
                for r in all_results for w in r["warnings"]]
if all_warnings:
    print(f"\n── Parser warnings ({len(all_warnings)} total) ──────────────────────")
    for name, warn in all_warnings[:20]:
        print(f"  [{name}]  {warn}")
    if len(all_warnings) > 20:
        print(f"  ... and {len(all_warnings)-20} more (see run_log.json)")


## Cell 9 — Build Master DataFrames

In [ ]:
# ── Flat profile table (one row per charity, latest FY) ──────────────
df_profiles = pd.DataFrame([r["profile"] for r in all_results])

# ── Long financials (all 3 years × all metrics) ──────────────────────
long_rows = []
for r in all_results:
    cid  = r["profile"]["charity_id"]
    name = r["profile"]["charity_name"]
    for f in r["financials_long"]:
        long_rows.append({"charity_id": cid, "charity_name": name, **f})
df_long = pd.DataFrame(long_rows) if long_rows else pd.DataFrame()

# ── Sofia upload (CI metrics only — 13 columns) ───────────────────────
SOFIA_COLS = [
    "charity_id", "charity_name", "fiscal_year_latest",
    "star_rating",
    "financial_transparency",
    "results_reporting_grade",
    "demonstrated_impact_rating",
    "need_for_funding_program_costs",
    "need_for_funding_reserve_funds",
    "need_for_funding_endowments",
    "cents_to_the_cause_pct",
    "scraped_at",
]
df_sofia = df_profiles[[c for c in SOFIA_COLS if c in df_profiles.columns]].copy()

print(f"df_profiles  :  {df_profiles.shape[0]:>4} rows  ×  {df_profiles.shape[1]:>3} columns")
print(f"df_long      :  {df_long.shape[0]:>4} rows  ×  {df_long.shape[1]:>3} columns")
print(f"df_sofia     :  {df_sofia.shape[0]:>4} rows  ×  {df_sofia.shape[1]:>3} columns")

print("\n── Field coverage ──────────────────────────────────────────────")
key_fields = [
    "star_rating", "fiscal_year_latest", "total_revenues",
    "cents_to_the_cause_pct", "avg_compensation_cad",
    "admin_cost_pct_revenue", "total_overhead_pct",
]
for f in key_fields:
    if f in df_profiles.columns:
        n   = df_profiles[f].notna().sum()
        pct = 100 * n / max(len(df_profiles), 1)
        bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"  {f:<40} [{bar}] {n}/{len(df_profiles)} ({pct:.0f}%)")


## Cell 10 — Data Quality Checks

In [ ]:
violations = []

def check(passed: bool, severity: str, column: str, rule: str, detail: str = ""):
    if not passed:
        violations.append({"severity":severity,"column":column,"rule":rule,"detail":detail})
        icon = "❌" if severity == "error" else "⚠️ "
        print(f"  {icon}  [{severity.upper():7}]  {column:<35}  {rule}  {detail}")

print("Running data-quality checks...\n")

n = len(df_profiles)
check(80 <= n <= 120, "error", "table", "rowCount",
      f"got {n} — update threshold in contract if set size changed")

if "charity_id" in df_profiles.columns:
    dupes = df_profiles["charity_id"].duplicated().sum()
    check(dupes == 0, "error", "charity_id", "no_duplicates", f"{dupes} duplicates")

if "star_rating" in df_profiles.columns:
    bad = df_profiles["star_rating"].dropna()
    out = int(((bad < 0) | (bad > 5)).sum())
    check(out == 0, "error", "star_rating", "range 0-5", f"{out} out-of-range")

if {"cents_to_the_cause_pct","total_overhead_pct"}.issubset(df_profiles.columns):
    both = df_profiles.dropna(subset=["cents_to_the_cause_pct","total_overhead_pct"])
    off  = int(((both.cents_to_the_cause_pct + both.total_overhead_pct - 100).abs() > 3).sum())
    check(off == 0, "warning", "cents+overhead", "sum ≈ 100%",
          f"{off} charities sum ≠ 100 (±3 tolerance)")

if "fiscal_year_latest" in df_profiles.columns:
    old = int((df_profiles["fiscal_year_latest"].dropna() < 2022).sum())
    check(old == 0, "warning", "fiscal_year_latest", "FY ≥ 2022",
          f"{old} charities show FY < 2022")

if "total_revenues" in df_profiles.columns:
    neg = int((df_profiles["total_revenues"].dropna() < 0).sum())
    check(neg == 0, "error", "total_revenues", "non-negative", f"{neg} negative values")

if not violations:
    print("✅  All checks passed.")
else:
    errs  = sum(1 for v in violations if v["severity"] == "error")
    warns = sum(1 for v in violations if v["severity"] == "warning")
    print(f"\nSummary: {errs} error(s), {warns} warning(s)")
    if errs:
        print("❌  Hard errors present — review before saving outputs.")

# ── Schema-drift radar ────────────────────────────────────────────
if not df_long.empty and "is_mapped" in df_long.columns:
    unmapped = (df_long[~df_long["is_mapped"]]
                .groupby("metric").size()
                .reset_index(name="n_charities")
                .sort_values("n_charities", ascending=False))
    if not unmapped.empty:
        print(f"\n── Unmapped financial labels (schema-drift radar) ──────────")
        print(unmapped.to_string(index=False))
        print("   → Add these to FIN_MAP / RATIO_MAP in Cell 5 if widespread")
    else:
        print("\n✅  No unmapped financial labels.")


## Cell 11 — Save Outputs

Writes: `ci_metrics.csv`, `ci_sofia_upload.csv`, `ci_financials_long.csv`, `WV_Competitive_Set.xlsx`, `run_log.json`

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

# ── Style helpers ─────────────────────────────────────────────────
NAVY   = "1B3A6B"
ORANGE = "E8680A"
GREY   = "F2F2F2"

def write_sheet(ws, df: pd.DataFrame, focal: str = "World Vision Canada"):
    """Write df to ws with styled header and WV pinned in orange."""
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)
    # Style header row
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
        cell.fill      = PatternFill("solid", fgColor=NAVY)
        cell.alignment = Alignment(wrap_text=True, horizontal="center")
    # Autofit columns
    for col_idx, col_name in enumerate(df.columns, 1):
        max_len = max(len(str(col_name)),
                      df[col_name].astype(str).str.len().max() if len(df) else 0)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 2, 28)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    # Highlight focal organisation row
    if focal and "charity_name" in df.columns:
        for i, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), 2):
            if row[0].value == focal:
                for cell in row:
                    cell.fill = PatternFill("solid", fgColor=ORANGE)
                    cell.font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)

# ── Sort: World Vision first, then alphabetical ──────────────────
FOCAL = "World Vision Canada"
wv_rows    = df_profiles[df_profiles.charity_name == FOCAL]
other_rows = df_profiles[df_profiles.charity_name != FOCAL].sort_values("charity_name")
df_sorted  = pd.concat([wv_rows, other_rows]).reset_index(drop=True)

# ── 1. CSVs ──────────────────────────────────────────────────────
df_sorted.to_csv(OUTPUT_DIR / "ci_metrics.csv", index=False)
df_sofia.to_csv(OUTPUT_DIR  / "ci_sofia_upload.csv", index=False)
if not df_long.empty:
    df_long.to_csv(OUTPUT_DIR / "ci_financials_long.csv", index=False)

# ── 2. Excel workbook ────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

def pick(cols):
    """Return columns that exist in df_sorted."""
    return [c for c in cols if c in df_sorted.columns]

# Tab 1: Overview
ws1 = wb.create_sheet("Overview")
write_sheet(ws1, df_sorted[pick([
    "charity_name","sector","fiscal_year_latest","year_end_month",
    "star_rating","cents_to_the_cause_pct","total_overhead_pct",
    "total_revenues","total_spending","full_time_staff",
    "avg_compensation_cad","profile_last_updated","profile_url",
])])

# Tab 2: Income
ws2 = wb.create_sheet("Income")
write_sheet(ws2, df_sorted[pick([
    "charity_name","fiscal_year_latest",
    "canadian_donations","international_donations","gifts_in_kind",
    "government_funding","special_events","fees_for_service",
    "investment_income","other_income","total_revenues",
])])

# Tab 3: Spending
ws3 = wb.create_sheet("Spending")
write_sheet(ws3, df_sorted[pick([
    "charity_name","fiscal_year_latest",
    "program_spending","fundraising_costs","administrative_costs",
    "grant_spending","gifts_in_kind_spending","other_costs",
    "program_spending_canada","program_spending_international",
    "total_spending","capital_spending","cash_flow_from_ops","funding_reserves",
])])

# Tab 4: Ratios
ws4 = wb.create_sheet("Ratios")
write_sheet(ws4, df_sorted[pick([
    "charity_name","fiscal_year_latest",
    "admin_cost_pct_revenue","fundraising_cost_pct_donations",
    "total_overhead_pct","cents_to_the_cause_pct",
    "program_cost_coverage_pct","fundraising_admin_cost_pct_revenue",
])])

# Tab 5: Staff
band_cols = [c for c in df_sorted.columns if c.startswith("comp_band_")]
ws5 = wb.create_sheet("Staff")
write_sheet(ws5, df_sorted[pick(
    ["charity_name","fiscal_year_latest","full_time_staff","avg_compensation_cad"]
    + band_cols
)])

# Tab 6: CI Ratings  ← the Sofia source
ws6 = wb.create_sheet("CI Ratings")
write_sheet(ws6, df_sorted[pick([
    "charity_name","fiscal_year_latest",
    "star_rating","financial_transparency","results_reporting_grade",
    "demonstrated_impact_rating",
    "need_for_funding_program_costs","need_for_funding_reserve_funds",
    "need_for_funding_endowments","cents_to_the_cause_pct",
    "is_top_100_rated","is_top_impact_charity",
])])

# Tab 7: Sofia Upload  ← paste into Sofia
ws7 = wb.create_sheet("Sofia Upload")
ws7.append([
    "⬆  Paste this tab into Sofia as the CI metrics source. "
    "Join on charity_name. Attribution: Charity Intelligence (charityintelligence.ca)"
])
ws7["A1"].font = Font(bold=True, color=ORANGE, name="Calibri", size=10)
ws7.row_dimensions[1].height = 25
ws7.append([])   # blank spacer row
df_sofia_sorted = pd.concat([
    df_sofia[df_sofia.charity_name == FOCAL],
    df_sofia[df_sofia.charity_name != FOCAL].sort_values("charity_name"),
]).reset_index(drop=True)
for row_data in dataframe_to_rows(df_sofia_sorted, index=False, header=True):
    ws7.append(row_data)
# Style header (row 3)
for cell in ws7[3]:
    cell.font = Font(bold=True, color="FFFFFF", name="Calibri", size=10)
    cell.fill = PatternFill("solid", fgColor=NAVY)

wb.save(OUTPUT_DIR / "WV_Competitive_Set.xlsx")

# ── 3. Run log ────────────────────────────────────────────────────
run_log = {
    "run_at"               : run_at,
    "parser_version"       : PARSER_VERSION,
    "charities_requested"  : len(work_list),
    "charities_parsed"     : len(all_results),
    "charities_errored"    : len(errors),
    "errors"               : errors,
    "dq_violations"        : violations,
    "all_parser_warnings"  : [(r["profile"].get("charity_name"), w)
                              for r in all_results for w in r["warnings"]],
}
(OUTPUT_DIR / "run_log.json").write_text(json.dumps(run_log, indent=2, default=str))

# ── Summary ───────────────────────────────────────────────────────
print("✅  All outputs saved\n")
print(f"  📊  WV_Competitive_Set.xlsx")
print(f"      Tabs   : {[ws.title for ws in wb.worksheets]}")
print(f"      Rows   : {len(df_sorted)} charities")
print(f"  📄  ci_metrics.csv           {df_sorted.shape[0]} × {df_sorted.shape[1]}")
print(f"  📄  ci_sofia_upload.csv      {df_sofia.shape[0]} × {df_sofia.shape[1]}")
if not df_long.empty:
    print(f"  📄  ci_financials_long.csv   {df_long.shape[0]} rows")
print(f"  📋  run_log.json")
print(f"\nAll files in:  {OUTPUT_DIR.resolve()}")


## Cell 11 — Preview the Results

In [ ]:
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.0f}".format)

print("=" * 70)
print("KEY METRICS — all charities")
print("=" * 70)
preview = [c for c in [
    "charity_name","fiscal_year_latest","star_rating",
    "total_revenues","cents_to_the_cause_pct",
    "total_overhead_pct","full_time_staff",
] if c in df_sorted.columns]
display(df_sorted[preview])

print("\n" + "=" * 70)
print("FINANCIAL RATIOS")
print("=" * 70)
ratio_preview = [c for c in [
    "charity_name","admin_cost_pct_revenue","fundraising_cost_pct_donations",
    "total_overhead_pct","cents_to_the_cause_pct","program_cost_coverage_pct",
] if c in df_sorted.columns]
if len(ratio_preview) > 1:
    display(df_sorted[ratio_preview])

print("\n" + "=" * 70)
print("SOFIA UPLOAD — 13 CI metrics")
print("=" * 70)
display(df_sofia)

# 3-year revenue trend
if not df_long.empty and "metric" in df_long.columns:
    rev = df_long[df_long["metric"] == "total_revenues"]
    if not rev.empty:
        print("\n" + "=" * 70)
        print("3-YEAR REVENUE TREND ($CAD)")
        print("=" * 70)
        pivot = (rev.pivot_table(
                     index="charity_name",
                     columns="fiscal_year",
                     values="value",
                     aggfunc="first")
                   .sort_values(by=rev["fiscal_year"].max()
                                if rev["fiscal_year"].notna().any() else
                                rev.columns[-1], ascending=False))
        display(pivot.style.format("${:,.0f}"))


## Cell 13 — Next Steps & Maintenance Guide

### After every run

| Step | Action |
|------|--------|
| 1 | Open `outputs/WV_Competitive_Set.xlsx` |
| 2 | Share with Magz for QA — she checks the Overview and CI Ratings tabs |
| 3 | Upload `outputs/ci_sofia_upload.csv` as the CI metrics source in Sofia (Sisense) |
| 4 | Check `outputs/run_log.json` for errors or DQ warnings |

### Adding or removing charities from the competitive set

Edit `COMPETITIVE_SET` in **Cell 1** and re-run from Cell 7.

```python
# Find the URL by visiting charityintelligence.ca and searching for the charity
# The URL pattern is always:  /charity-details/{id}-{slug}
{"charity_name": "New Charity",
 "profile_url": "https://www.charityintelligence.ca/charity-details/999-new-charity"},
```

### Unmapped financial labels (Cell 10)

After the first full run, any new line items CI introduces appear in the
schema-drift radar. Add them to `FIN_MAP` or `RATIO_MAP` in Cell 5,
then re-run from Cell 5.

### Automating with GitHub Actions

This notebook is equivalent to the `src/refresh.py` script in the GitHub repo.
The GitHub Action (`weekly_refresh.yml`) runs it automatically every Monday at 6 AM ET.
See `README.md` in the repo for the one-time secret setup.

### Refreshing the HTML cache

The client caches pages for 7 days (`MAX_AGE_DAYS`).
To force a fresh fetch of everything, delete `cache/html/` and re-run.

```python
# Or change the constant at the top of Cell 1:
MAX_AGE_DAYS = 0   # always re-fetch
```

### Attribution
> **Source: Charity Intelligence (charityintelligence.ca)**  
> Always include this line when sharing any output from this notebook.
